# 16_01 — Simulación y preprocesamiento · Escenario 6 (Algoritmo 6 del anexo)

Paso **1 de 3** del ciclo `Python → MATLAB → Python`:

| Paso | Archivo | Qué hace |
|---|---|---|
| 1 | **este notebook** | genera los datos, ajusta base + FPCA + estandarizador, escribe los datasets AR y todos los artefactos |
| 2 | `psbp_fd_iteracion.m` | muestreo MCMC en MATLAB, sólo con el bloque de entrenamiento |
| 3 | `16_03_convergencia` / `16_04_evaluacion` | diagnóstico de cadenas y evaluación fuera de muestra |

Las celdas marcadas **`[CONFIG]`** son las únicas que se tocan al cambiar de
experimento. Todo lo demás se deriva de ellas.

## 1. Imports y rutas

In [2]:
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Generadores y contrato de artefactos
from model_psbp_fd.pipelines import (
    ConfigEscenario6, generar_escenario_6, guardar_escenario,
    guardar_curvas, guardar_representacion, guardar_fpca,
    guardar_estandarizador, guardar_datasets_ar,
    guardar_hiperparametros, guardar_config_evaluacion,
    verificar_contrato,
)
# Preprocesamiento funcional
from model_psbp_fd.functions_models import (
    FunctionalRepresentation, FPCA_L2, base_en_grilla, DataStandardizer,
)
from model_psbp_fd.fit import tabla_baselines
from model_psbp_fd.utils import get_project_root
from model_psbp_fd.utils.quadrature import pesos_trapezoidales
from model_psbp_fd.graphics import (
    plot_empirical_sample, plot_mean_and_variance, plot_fts_empirical,
    plot_fts_functional, plot_diagnostico_estandarizacion, plot_fpca_scree,
    plot_seleccion_basis, plot_rezagos_heatmap,
)

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

### 1.1 `[CONFIG]` Identificación del experimento

In [3]:
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASENAME     = "escenario"
ESCENARIO_ID = 6      # Algoritmo k del anexo
REPLICA_ID   = 1      # réplica Monte Carlo; eje del barrido en la Etapa D
SEED         = 41232  # semilla base; MATLAB la LEE de hyperparameters.json

EXPERIMENT_ID = f"{BASENAME}_{ESCENARIO_ID}_r{REPLICA_ID:02d}"

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"EXPERIMENT_ID : {EXPERIMENT_ID}")
print(f"Escenario {ESCENARIO_ID} · réplica {REPLICA_ID} · seed base {SEED}")

PROJECT_ROOT  : C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1
EXPERIMENT_ID : escenario_6_r01
Escenario 6 · réplica 1 · seed base 41232


In [4]:
# Las cinco rutas del contrato. No existe un config_paths en Python: cada
# notebook lo arma aquí y config_paths.m replica las mismas del lado MATLAB.
PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / EXPERIMENT_ID,
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / EXPERIMENT_ID,
    "out_report":   PROJECT_ROOT / "reports" / "simulaciones" / EXPERIMENT_ID,
    "out_artefact": PROJECT_ROOT / "artefact" / "simulaciones" / EXPERIMENT_ID,
}
for nombre, ruta in PATHS.items():
    ruta.mkdir(parents=True, exist_ok=True)
    print(f"  {nombre:12s} → {ruta}")

  raw          → C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\data\simulaciones\raw\escenario_6_r01
  functional   → C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\data\simulaciones\processed\functional\escenario_6_r01
  predict      → C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\data\simulaciones\processed\predict\escenario_6_r01
  out_report   → C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\reports\simulaciones\escenario_6_r01
  out_artefact → C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\artefact\simulaciones\escenario_6_r01


## 2. Simulación

### 2.1 `[CONFIG]` Parámetros del generador


In [5]:
# ── Parámetros fijos del estudio (comunes a los 6 escenarios) ────────────────
L_GRILLA   = 75     # puntos de la grilla regular tau_1=0 … tau_L=1
T_CURVAS   = 400    # curvas retenidas tras el calentamiento
PROP_TRAIN = 0.70   # proporción del bloque de entrenamiento
SIGMA_OBS  = 0.25   # desviación del ruido de medición

T0_PREVIO  = int(np.floor(PROP_TRAIN * T_CURVAS))   # 280; se recalcula en §2.5
T_QUIEBRE  = T0_PREVIO   # ← DECISIÓN de esta corrida; el default del código es 270

def media_senoidal(tau):
    """mu(tau) = sin(2 pi tau), Cuadro tab:ane_esquema. Con nombre para que
    quede legible en el JSON."""
    return np.sin(2.0 * np.pi * tau)

SIM_CFG = ConfigEscenario6(
    # Esquema de observación. TODOS explícitos: los defaults de esta dataclass
    # son los del Cuadro del Capítulo 3 (L=48, T=300, sigma_obs=0.1, R=50) y
    # NO son los del estudio.
    L         = L_GRILLA,
    T         = T_CURVAS,
    burn_in   = 200,
    sigma_obs = SIGMA_OBS,
    R         = 1,            # una réplica por corrida; el barrido usa REPLICA_ID
    seed      = SEED,
    media_fn  = media_senoidal,
    # Sistema ortonormal y espectros (Cuadro tab:ane_alg6)
    J                   = 10,
    razon_espectro      = 0.5,      # lambda^(0)_j = 0.5^(j-1)
    lambdas_inicial     = None,     # None ⇒ espectro geométrico
    lambdas_final       = None,     # None ⇒ intercambio de indices_intercambio
    indices_intercambio = (1, 2),   # BASE-1
    # Dinámica: phi COMÚN, para aislar el reordenamiento del espectro
    phis      = None,   # None ⇒ phi_j = phi_comun en todas
    phi_comun = 0.7,
    # Trayectoria del espectro
    modo      = "quiebre",
    t_quiebre = T_QUIEBRE,   # ← T0, no el 270 del default. Ver la cabecera.
)

for k, v in SIM_CFG.to_dict().items():
    print(f"  {k:<24}: {v}")

# Verificación explícita de que los defaults NO se colaron.
assert (SIM_CFG.L, SIM_CFG.T, SIM_CFG.sigma_obs, SIM_CFG.R) == (75, 400, 0.25, 1), \
    ("Algún parámetro del esquema de observación quedó con el default de "
     "ConfigEscenario6 (L=48, T=300, sigma_obs=0.1, R=50) en vez del valor del "
     "estudio.")
assert SIM_CFG.t_quiebre == T0_PREVIO, \
    f"t_quiebre={SIM_CFG.t_quiebre} != T0={T0_PREVIO}: revisar la decisión de §2.1."
print(f"\n  Esquema de observación sobrescrito correctamente.")
print(f"  t* = {SIM_CFG.t_quiebre} = T0   ·   periodos post-quiebre: "
      f"{T_CURVAS - SIM_CFG.t_quiebre + 1}")

  L                       : 75
  T                       : 400
  burn_in                 : 200
  sigma_obs               : 0.25
  R                       : 1
  seed                    : 41232
  media_fn                : media_senoidal
  jitter                  : 1e-10
  J                       : 10
  lambdas_inicial         : None
  lambdas_final           : None
  razon_espectro          : 0.5
  indices_intercambio     : (1, 2)
  phis                    : None
  phi_comun               : 0.7
  modo                    : quiebre
  t_quiebre               : 280

  Esquema de observación sobrescrito correctamente.
  t* = 280 = T0   ·   periodos post-quiebre: 121


### 2.2 Generación

In [6]:
salida = generar_escenario_6(SIM_CFG)

REPLICA_IDX = REPLICA_ID - 1
X_raw   = salida.observaciones[REPLICA_IDX]   # (T, G) OBSERVADA — alimenta la estimación
X_true  = salida.curvas[REPLICA_IDX]          # (T, G) VERDADERA — objetivo de evaluación
A_coef  = salida.internos["coeficientes"][REPLICA_IDX]   # (T, J) coeficientes verdaderos
PHI_GEN = salida.internos["base"]                        # (G, J) base de Fourier
LAM0    = salida.internos["espectro_inicial"]            # (J,)
LAM1    = salida.internos["espectro_final"]              # (J,)
LAM_T   = salida.internos["trayectoria_espectro"]        # (T, J)
PHIS    = salida.internos["coeficientes_ar"]             # (J,)
grilla  = salida.grilla
T, G    = X_raw.shape
J_GEN   = int(SIM_CFG.J)

print(f"Observadas {X_raw.shape} · verdaderas {X_true.shape} · grilla {grilla.shape}")
print(f"coeficientes {A_coef.shape} · trayectoria del espectro {LAM_T.shape}")
print(f"Ruido de medición efectivo: sd(X_raw - X_true) = {(X_raw - X_true).std():.4f}"
      f"   (nominal {SIGMA_OBS})")
print("\nControl de calidad del generador:")
for k, v in salida.diagnostico.items():
    if isinstance(v, list) and len(v) > 6:
        v = [round(float(x), 5) for x in v]
    print(f"  {k:36s} = {v}")

Observadas (400, 75) · verdaderas (400, 75) · grilla (75,)
coeficientes (400, 10) · trayectoria del espectro (400, 10)
Ruido de medición efectivo: sd(X_raw - X_true) = 0.2513   (nominal 0.25)

Control de calidad del generador:
  n_replicas                           = 1
  n_curvas                             = 400
  n_puntos_grilla                      = 75
  todo_finito                          = True
  media_empirica_global                = -0.17904345908759237
  media_ee_montecarlo                  = nan
  var_puntual_media                    = 1.9862216076607917
  var_puntual_min                      = 1.3969780265885825
  var_puntual_max                      = 2.8726688113406285
  acf1_media                           = 0.6998579188042929
  acf1_ee_montecarlo                   = nan
  razon_senal_ruido                    = 31.779545722572667
  J                                    = 10
  modo_trayectoria                     = quiebre
  t_quiebre_objetivo                   = 280
  per

In [7]:
simulation_config = {
    "sim_params":    salida.config.to_dict(),
    "diagnostico":   salida.diagnostico,
    "replica_idx":   REPLICA_IDX,
    "experiment_id": EXPERIMENT_ID,
    "escenario_id":  int(ESCENARIO_ID),
    "replica_id":    int(REPLICA_ID),
    "seed":          SEED,
    "T": int(T), "G": int(G), "J": int(J_GEN),
    "t_quiebre":     int(TQ),
    "t_quiebre_default_del_codigo": 270,
    "motivo_t_quiebre": (
        "Reubicado en T0 = 280: el default 270 está dimensionado para T=300 / "
        "T0=240. Con t* = T0 el entrenamiento queda homogéneo, todo el bloque "
        "de prueba queda post-quiebre y la ventana posterior pasa de 30 a 120 "
        "periodos, con lo que reordenamiento_detectado deja de ser poco fiable "
        "con R = 1."
    ),
}
with open(PATHS["raw"] / "simulation_config.json", "w", encoding="utf-8") as f:
    json.dump(simulation_config, f, indent=2, ensure_ascii=False)

# incluir_internos=True guarda `interno_coeficientes` (los a_tj verdaderos),
# `interno_trayectoria_espectro` (el estado verdadero), los dos espectros y los
# phi. Ninguno entra en la estimación.
_npz = guardar_escenario(salida, str(PATHS["raw"] / f"escenario_{ESCENARIO_ID}"),
                         incluir_curvas=True, incluir_internos=True)
print(f"[raw] simulation_config.json  ·  {_npz}")

NameError: name 'TQ' is not defined

#### Lectura del control de calidad

Las verificaciones que este escenario necesita, y cuáles son criterio y cuáles
no:

**ESTRUCTURALES** (si fallan, el generador no es el del anexo):

- **`base_ortonormal`** y **`reproyeccion_error_max`**: sin esas dos identidades
  el reordenamiento del espectro no sería observable sobre las curvas.
- **`reordenamiento_esperado`**: que los espectros inicial y final tengan órdenes
  distintos. Es una propiedad de la configuración, no de la realización.
- **`orden_pre_correcto`**: que la ventana previa al quiebre reproduzca el orden
  de $\lambda^{(0)}$. Con 279 períodos y $R=1$ es una comprobación con potencia
  suficiente.

**INFORMATIVOS** (cifras ruidosas con $R=1$; se reportan, no se exigen):

- **`orden_post_correcto`** compara el orden completo de las 10 componentes en la
  ventana posterior. Las componentes altas tienen varianzas del orden de
  $10^{-3}$, indistinguibles entre sí con una sola trayectoria, de modo que un
  intercambio espurio entre la 8 y la 9 basta para que salga `False` sin que
  nada esté mal. **Lo que importa es el orden de las dos componentes
  intercambiadas**, y se verifica por separado abajo.
- **`reordenamiento_detectado`** compara los órdenes empírico pre y post. El
  módulo advierte que con la configuración del anexo —ventana post de 30
  períodos— puede dar `False` con $R=1$ pese a que el generador es correcto. Con
  $t^{*}=T_0$ la ventana pasa a 120 períodos y el diagnóstico gana potencia,
  que es una de las razones de la decisión.
- **`instante_quiebre_estimado`**: el CUSUM sobre el contraste de energía entre
  las dos componentes intercambiadas. Un error de unos pocos períodos es
  esperable, del orden de `periodos_adaptacion`: la varianza marginal no salta,
  converge al nuevo nivel como $\varphi^{2n}$.

In [ ]:
_d = salida.diagnostico
_i1, _i2 = int(SIM_CFG.indices_intercambio[0]), int(SIM_CFG.indices_intercambio[1])

# ── ESTRUCTURALES ───────────────────────────────────────────────────────────
_estructurales = [
    ("trayectorias finitas",              _d["todo_finito"],
     f"{_d['n_replicas']}x{_d['n_curvas']}x{_d['n_puntos_grilla']}"),
    ("base ortonormal en la métrica L2",  _d["base_ortonormal"],
     f"error max {_d['ortonormalidad_error_max']:.3e}"),
    ("reproyeccion exacta de A",          _d["reproyeccion_error_max"] < 1e-8,
     f"error max {_d['reproyeccion_error_max']:.3e}"),
    ("reordenamiento ESPERADO por diseño", _d["reordenamiento_esperado"],
     f"orden objetivo pre {_d['orden_objetivo_pre'][:4]} → "
     f"post {_d['orden_objetivo_post'][:4]}"),
    ("orden pre-quiebre correcto",        _d["orden_pre_correcto"],
     f"empírico {_d['orden_componentes_pre'][:4]}"),
]
for nombre, ok, detalle in _estructurales:
    print(f"  {'OK ' if ok else 'FALLA'}  {nombre:36s} {detalle}")

assert all(ok for _, ok, _ in _estructurales), \
    "El generador no cumple las condiciones estructurales del Algoritmo 6."

# La comprobación que SÍ tiene potencia: el orden de las DOS componentes
# intercambiadas, no el de las diez. Es la que decide si el escenario existe.
_vpre  = np.asarray(_d["var_empirica_pre"])
_vpost = np.asarray(_d["var_empirica_post"])
_swap_pre  = _vpre[_i1 - 1]  > _vpre[_i2 - 1]     # antes: la i1 domina
_swap_post = _vpost[_i1 - 1] < _vpost[_i2 - 1]    # después: la i2 domina
print(f"\n  {'OK ' if _swap_pre and _swap_post else 'FALLA'}  "
      f"intercambio efectivo de las componentes {_i1} y {_i2}")
print(f"      pre : var[{_i1}]={_vpre[_i1-1]:.5f} vs var[{_i2}]={_vpre[_i2-1]:.5f}"
      f"   → domina la {_i1 if _swap_pre else _i2}")
print(f"      post: var[{_i1}]={_vpost[_i1-1]:.5f} vs var[{_i2}]={_vpost[_i2-1]:.5f}"
      f"   → domina la {_i2 if _swap_post else _i1}")
assert _swap_pre and _swap_post, (
    "Las componentes intercambiadas no cambiaron de orden en la realización: "
    "el escenario no tiene contenido. Revisar t_quiebre o la semilla.")

In [ ]:
# ── INFORMATIVOS: con R = 1 son cifras ruidosas ─────────────────────────────
print("Diagnósticos informativos (R = 1; se reportan, no son criterio):")
print(f"  reordenamiento_detectado          : {_d['reordenamiento_detectado']}")
print(f"  orden_post_correcto (las 10 comp.): {_d['orden_post_correcto']}")
print(f"      empírico post {_d['orden_componentes_post']}")
print(f"      objetivo post {_d['orden_objetivo_post']}")
if not _d["orden_post_correcto"]:
    _dif = [(a, b) for a, b in zip(_d["orden_componentes_post"],
                                   _d["orden_objetivo_post"]) if a != b]
    print(f"      discrepan en {len(_dif)} posiciones: {_dif}")
    print("      Con R = 1 las componentes altas tienen varianzas del orden de "
          "1e-3 y son\n      indistinguibles entre sí: un intercambio espurio "
          "entre ellas basta para\n      que esta bandera salga False. El "
          "criterio que manda es el intercambio de\n      las componentes "
          f"{_i1} y {_i2}, verificado arriba.")

print(f"\n  periodos_adaptacion               : {_d['periodos_adaptacion']}   "
      f"(la varianza converge como phi^(2n))")
print(f"  instante_quiebre_estimado (CUSUM) : {_d['instante_quiebre_estimado']}   "
      f"objetivo {_d['t_quiebre_objetivo']}   "
      f"error {_d['instante_quiebre_error']}")
print(f"  espectro_pre_error_max            : {_d['espectro_pre_error_max']:.5f}")
print(f"  espectro_post_error_max           : {_d['espectro_post_error_max']:.5f}")
print(f"\n  acf1 empírica por componente (objetivo phi = {SIM_CFG.phi_comun}):")
print(f"      total     {[round(float(a), 3) for a in _d['acf1_por_componente']]}")
print(f"      pre-quiebre {[round(float(a), 3) for a in _d['acf1_por_componente_pre']]}")
print("      La variación del espectro sesga levemente a la baja la acf1 "
      "calculada sobre\n      la serie completa; por eso se reporta también "
      "la de la ventana previa.")

_ventana_post = T - SIM_CFG.t_quiebre + 1
print(f"\n  ventana post-quiebre: {_ventana_post} periodos "
      f"(con el default t*=270 y T=300 serían 30)")
print("      Es la razón dimensional de fijar t* = T0: el diagnóstico de "
      "reordenamiento\n      gana potencia y deja de ser poco fiable con R = 1.")

#### El rasgo del escenario: el espectro se reordena

Figura propia del Algoritmo 6. Muestra el mecanismo completo: la trayectoria
determinista del espectro, la varianza empírica de las dos componentes
intercambiadas a lo largo del tiempo, y los dos órdenes.

La figura se dibuja sobre los **coeficientes verdaderos** $a_{tj}$ en la base de
Fourier del generador, no sobre los scores FPCA. Que el reordenamiento sobreviva
al pipeline —base B-spline, FPCA ajustado en entrenamiento— es una pregunta
distinta, y se responde en §3.3.

In [ ]:
w_quad  = pesos_trapezoidales(grilla)
TQ      = int(SIM_CFG.t_quiebre)          # base-1
pre     = np.arange(0, TQ - 1)            # base-0: t = 1 .. TQ-1
post    = np.arange(TQ - 1, T)            # base-0: t = TQ .. T
N_ADAPT = int(_d["periodos_adaptacion"])

colores = ["#c0392b", "#2980b9"]
fig = plt.figure(figsize=(13.5, 7.2))
gs  = fig.add_gridspec(2, 2, width_ratios=[3, 1.15], hspace=0.42, wspace=0.28)

# (a) las dos componentes intercambiadas a lo largo del tiempo
ax = fig.add_subplot(gs[0, 0])
for c, i in enumerate((_i1, _i2)):
    ax.plot(np.arange(1, T + 1), A_coef[:, i - 1], lw=0.7, alpha=0.8,
            color=colores[c], label=rf"$a_{{t,{i}}}$")
ax.axvline(TQ, color="k", ls="--", lw=1.4)
ax.text(TQ, ax.get_ylim()[1], r" $t^*=T_0$", fontsize=9, va="top")
ax.set_ylabel("coeficiente"); ax.legend(fontsize=8, ncol=2)
ax.set_title("Las dos componentes intercambiadas: la amplitud se invierte "
             "en $t^*$", fontsize=10)

# (b) varianza empírica en ventana móvil de las dos componentes
ax = fig.add_subplot(gs[0, 1])
_wv = 40
_v = np.array([[A_coef[max(0, t - _wv):t, i - 1].var() for i in (_i1, _i2)]
               for t in range(_wv, T + 1)])
for c in range(2):
    ax.plot(np.arange(_wv, T + 1), _v[:, c], lw=1.2, color=colores[c])
ax.axvline(TQ, color="k", ls="--", lw=1.2)
ax.set_title(f"varianza móvil (w={_wv})", fontsize=9)

# (c) trayectoria determinista del espectro
ax = fig.add_subplot(gs[1, 0])
for c, i in enumerate((_i1, _i2)):
    ax.plot(np.arange(1, T + 1), LAM_T[:, i - 1], lw=1.6, color=colores[c],
            label=rf"$\lambda_{{{i}}}(t)$")
ax.axvline(TQ, color="k", ls="--", lw=1.4)
ax.axvspan(TQ, min(T, TQ + N_ADAPT), color="0.75", alpha=0.5)
ax.text(TQ + N_ADAPT, ax.get_ylim()[1], f"  adaptación ({N_ADAPT})",
        fontsize=8, va="top")
ax.set_xlabel("$t$"); ax.set_ylabel(r"$\lambda_j(t)$"); ax.legend(fontsize=8)
ax.set_title("Trayectoria determinista del espectro (el salto es del "
             "PARÁMETRO)", fontsize=10)

# (d) los dos espectros
ax = fig.add_subplot(gs[1, 1])
_jx = np.arange(1, J_GEN + 1)
ax.plot(_jx, LAM0, "o-", ms=4, color="#2c3e50", label=r"$\lambda^{(0)}$")
ax.plot(_jx, LAM1, "s--", ms=4, color="#e67e22", label=r"$\lambda^{(1)}$")
ax.set_yscale("log"); ax.set_xticks(_jx[::2])
ax.set_xlabel("$j$"); ax.legend(fontsize=8)
ax.set_title("el final NO es decreciente", fontsize=9)

fig.suptitle("Escenario 6 — la covarianza se reordena en $t^*=T_0$: la base "
             "ajustada en entrenamiento pierde vigencia", fontsize=12)
fig.savefig(PATHS["out_report"] / "10_quiebre_espectro.png", dpi=150,
            bbox_inches="tight")
plt.show()

print(f"varianza empírica de las componentes {_i1} y {_i2}:")
print(f"  pre  (t < {TQ}, n={pre.size}) : "
      f"{A_coef[pre, _i1-1].var():.5f}  vs  {A_coef[pre, _i2-1].var():.5f}")
print(f"  post (t >= {TQ + N_ADAPT}, n={T - TQ - N_ADAPT + 1}) : "
      f"{A_coef[TQ-1+N_ADAPT:, _i1-1].var():.5f}  vs  "
      f"{A_coef[TQ-1+N_ADAPT:, _i2-1].var():.5f}")

#### Persistencia del estado verdadero

A diferencia de las corridas 14 y 15, este escenario **sí tiene un estado
latente por el cual estratificar**, y además es **discreto**: pre o post
quiebre. Se pasa directo a `fit.cobertura_condicional`, sin cuantilizar, igual
que el régimen en la corrida 13.

Se guarda como CSV —y no sólo dentro del `.npz`— porque es el insumo del eje 2
de la evaluación y conviene que sea legible sin abrir un binario.

**Repetir aquí la advertencia**: con $t^{*}=T_0$ este estrato coincide con la
partición train/test. La columna `en_adaptacion` marca los `periodos_adaptacion`
posteriores al quiebre, en los que la varianza marginal todavía converge al
nuevo nivel: no son ni un régimen ni el otro, y conviene poder excluirlos.

In [ ]:
en_post   = np.arange(1, T + 1) >= TQ
en_adapt  = (np.arange(1, T + 1) >= TQ) & (np.arange(1, T + 1) < TQ + N_ADAPT)
nivel     = (X_true * w_quad).sum(axis=1)          # <X_t, 1>
energia   = (X_true ** 2) @ w_quad                 # ||X_t||^2, sensible al espectro

estado_df = pd.DataFrame({
    "t":              np.arange(1, T + 1),
    "quiebre":        en_post.astype(int) + 1,   # base-1: 1 = pre, 2 = post
    "quiebre_idx":    en_post.astype(int),       # base-0: el que consume cobertura_condicional
    "en_adaptacion":  en_adapt,
    "nivel_curva":    nivel,
    "energia_curva":  energia,
})
for i in (_i1, _i2):
    estado_df[f"a_fourier_{i}"] = A_coef[:, i - 1]
estado_df.to_csv(PATHS["out_report"] / "10_estado_quiebre.csv", index=False)
print(f"[out_report] 10_estado_quiebre.csv  {estado_df.shape}")
print(f"  pre-quiebre : {int((~en_post).sum())} periodos")
print(f"  post-quiebre: {int(en_post.sum())} periodos "
      f"(de ellos {int(en_adapt.sum())} en adaptación)")
display(estado_df.head())

# Verificación numérica del estado reconstruido contra la trayectoria que el
# generador registró: el estrato debe coincidir con LAM_T fila a fila.
_post_por_lambda = np.array([not np.allclose(LAM_T[t], LAM0) for t in range(T)])
assert np.array_equal(_post_por_lambda, en_post), (
    "El estrato pre/post no coincide con internos['trayectoria_espectro']: "
    "revisar la convención base-1 de t_quiebre.")
print("\n  El estrato reconstruido coincide fila a fila con la trayectoria del "
      "espectro\n  que el generador registró en internos.")

### 2.3 Visualización de los datos

In [ ]:
highlight_idx = [0, 1, T // 2, T - 1]

plot_fts_empirical(
    X_raw, grilla, highlight_idx=highlight_idx, separator_every=5,
    title=f"Covarianza no estacionaria — {T} curvas observadas",
    save_path=str(PATHS["out_report"] / "01_fts_empirica_raw.png"))
plt.show()

plot_empirical_sample(
    X_raw, grilla, sample_idx=[0, 40, 80, TQ + 20, T - 1],
    title="Muestra de 5 curvas observadas (una posterior al quiebre)",
    save_path=str(PATHS["out_report"] / "02_muestra_empirica_raw.png"))
plt.show()

plot_mean_and_variance(
    X_raw, grilla, show_std1=True, show_std2=True,
    title="Media y varianza funcional — Algoritmo 6",
    save_path=str(PATHS["out_report"] / "03_media_varianza_raw.png"))
plt.show()

# Varianza funcional ANTES y DESPUÉS: la firma del escenario sobre las curvas.
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(grilla, X_true[pre].var(axis=0), lw=1.8, color="#2c3e50",
        label=f"pre-quiebre (n={pre.size})")
ax.plot(grilla, X_true[TQ-1+N_ADAPT:].var(axis=0), lw=1.8, color="#e67e22",
        ls="--", label=f"post-quiebre (n={T - TQ - N_ADAPT + 1})")
ax.set_xlabel(r"$\tau$"); ax.set_ylabel("varianza puntual"); ax.legend(fontsize=8)
ax.set_title("Varianza puntual antes y después del quiebre — el reordenamiento "
             "cambia la FORMA, no sólo la escala", fontsize=10)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "11_varianza_pre_post.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
# Curva observada vs verdadera: dimensiona el ruido que el modelo NO debe predecir
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for ax, i in zip(axes, [0, TQ // 2, T - 1]):
    ax.plot(grilla, X_raw[i], ".", color="0.65", ms=3, label="observada (con ruido)")
    ax.plot(grilla, X_true[i], color="#c0392b", lw=1.6, label="verdadera $X_t(\\tau)$")
    ax.set_title(rf"$t={i+1}$  ({'post' if i + 1 >= TQ else 'pre'}-quiebre)",
                 fontsize=10)
    ax.set_xlabel(r"$\tau$")
axes[0].set_ylabel(r"$X_t(\tau)$"); axes[0].legend(fontsize=8)
fig.suptitle("Curva verdadera vs datos observados — el error se mide contra la primera",
             fontsize=12)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "04_curva_vs_datos.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.4 Persistencia de curvas

Se guardan las dos matrices con nombres distintos (`X_curves.npy` y
`X_curves_true.npy`) para que no puedan confundirse aguas abajo.

In [ ]:
_p = guardar_curvas(PATHS, X_raw, grilla, X_true=X_true)
for clave, ruta in _p.items():
    print(f"[functional] {clave:12s} → {ruta.name}")

### 2.5 Partición temporal — y la coincidencia con el quiebre

Todo objeto **estimado a partir de los datos** —selección GCV de la base, FPCA,
estandarizador— se ajusta sólo con $\{1,\dots,T_0\}$ y se aplica al bloque de
prueba mediante `transform`.

**Y aquí es donde la decisión de $t^{*}=T_0$ tiene su consecuencia.** El bloque
de entrenamiento es homogéneo —un solo régimen de covarianza— de modo que la
base FPCA está bien definida; y todo el bloque de prueba pertenece al otro
régimen, de modo que esa base ya no es la correcta. Eso es exactamente la tesis
del escenario, y también el motivo por el que el estrato del eje 2 coincide con
la partición.

In [ ]:
T0 = int(np.floor(PROP_TRAIN * T))
assert 10 < T0 < T, f"T0={T0} fuera de rango para T={T}."
assert T0 == TQ, (f"T0={T0} y t_quiebre={TQ} deberían coincidir en esta corrida; "
                  "si se cambia uno hay que revisar §9 de 16_04.")

idx_train, idx_test = np.arange(0, T0), np.arange(T0, T)
X, X_train, X_test = X_raw, X_raw[idx_train], X_raw[idx_test]

print(f"entrenamiento : t en [1, {T0}]      → {X_train.shape}")
print(f"prueba        : t en [{T0+1}, {T}]  → {X_test.shape}")
print(f"proporción    : {T0/T:.1%} / {1 - T0/T:.1%}")

# El período t = t* es el PRIMERO generado con el espectro final, y con
# t* = T0 cae dentro del bloque de entrenamiento. Es un único período de 280.
_n_post_en_train = int(en_post[idx_train].sum())
print(f"\nperíodos post-quiebre dentro del ENTRENAMIENTO: {_n_post_en_train} "
      f"de {T0}")
print(f"períodos post-quiebre dentro de la PRUEBA      : "
      f"{int(en_post[idx_test].sum())} de {T - T0}")
assert _n_post_en_train <= 1, (
    "Más de un período post-quiebre cayó en entrenamiento: el bloque dejaría "
    "de ser homogéneo.")

print("\n" + "=" * 70)
print("ADVERTENCIA que hay que trasladar al reporte:")
print("""
  Con t* = T0 el estrato pre/post quiebre COINCIDE con la partición
  train/test. La cobertura condicional de 16_04 §9 y la comparación
  train/test de §4 miden por tanto LO MISMO.

  No es un defecto: es la tesis del escenario. Pero sin declararlo se
  leería el desplome de cobertura en el bloque de prueba como pérdida de
  GENERALIZACIÓN, cuando es pérdida de VIGENCIA DE LA BASE. La primera se
  arregla con más datos; la segunda, sólo reajustando la representación.

  Para separar ambos efectos habría que mover t* dentro del bloque de
  prueba (p. ej. 340), aceptando una ventana post-quiebre más corta.""")
print("=" * 70)

# Cuánto cambia el espectro empírico entre bloques: la cifra que anticipa §9.
_v_tr = A_coef[idx_train].var(axis=0)
_v_te = A_coef[idx_test].var(axis=0)
print(f"\nvarianza de los coeficientes verdaderos, por bloque:")
print(f"  {'j':>3} {'train':>9} {'test':>9} {'razón':>7}")
for j in range(min(5, J_GEN)):
    marca = "  <- intercambiada" if (j + 1) in (_i1, _i2) else ""
    print(f"  {j+1:>3} {_v_tr[j]:>9.5f} {_v_te[j]:>9.5f} "
          f"{_v_te[j]/_v_tr[j]:>7.2f}{marca}")
print("   En el Algoritmo 5 estas razones eran todas cercanas a 1. Aquí las dos "
      "primeras\n   se invierten: eso es lo que hace obsoleta la base.")

## 3. Representación funcional B-spline

### 3.1 Barrido GCV sobre `(n_basis, order)`

El GCV **sugiere**; la elección es del analista y se declara en 3.2. Como en el
Algoritmo 5, el proceso vive en el span de $J=10$ funciones de Fourier y la base
B-spline del estudio introduce un error de representación **adicional** al del
truncamiento FPCA. `16_04` los separa.

Nota: el barrido se hace **sólo con entrenamiento**, que es homogéneo. La base
elegida es por tanto la óptima para el régimen viejo — que es precisamente la
situación que el escenario reproduce.

In [ ]:
N_BASIS_RANGE = range(2, min(30, T0 // 2))   # acotado por T0
ORDER_RANGE   = range(2, 5)

registros = []
for orden in ORDER_RANGE:
    for nb in N_BASIS_RANGE:
        if nb < orden:
            continue
        try:
            fr_tmp = FunctionalRepresentation(method="bspline", n_basis=nb, order=orden)
            TH_tmp = fr_tmp.fit_transform(X_train, grilla)     # sólo train
            X_rec  = fr_tmp.reconstruct(TH_tmp)

            L_i    = X_train.shape[1]
            sse_c  = np.sum((X_train - X_rec) ** 2, axis=1)
            ss_tot = np.sum((X_train - X_train.mean(axis=0, keepdims=True)) ** 2)
            gcv_c  = (L_i * sse_c / (L_i - nb) ** 2 if L_i > nb
                      else np.full_like(sse_c, np.nan))
            registros.append({
                "n_basis": nb, "order": orden,
                "var_retained": 1.0 - sse_c.sum() / ss_tot,
                "rmse_mean": np.sqrt(sse_c / L_i).mean(),
                "rmse_max":  np.sqrt(sse_c / L_i).max(),
                "gcv_mean":  float(np.mean(gcv_c)),
            })
        except Exception as e:
            print(f"  [SKIP] n_basis={nb}, order={orden}: {e}")

sel_df   = pd.DataFrame(registros)
best_row = sel_df.dropna(subset=["gcv_mean"]).nsmallest(1, "gcv_mean").iloc[0]
nb_best, ord_best = int(best_row["n_basis"]), int(best_row["order"])

display(sel_df.style
    .format({"var_retained": "{:.4%}", "rmse_mean": "{:.6f}",
             "rmse_max": "{:.6f}", "gcv_mean": "{:.6f}"})
    .background_gradient(subset=["gcv_mean"], cmap="YlOrRd_r")
    .background_gradient(subset=["var_retained"], cmap="YlGn"))

print(f"\nGCV mínimo → n_basis={nb_best}, order={ord_best}  "
      f"(var retenida {best_row['var_retained']:.4%})")

plot_seleccion_basis(sel_df, nb_best, ord_best,
                     save_path=str(PATHS["out_report"] / "05_seleccion_basis.png"))
plt.show()

### 3.2 `[CONFIG]` Base elegida y ajuste

`center=False` es imprescindible: con `center=True` la reconstrucción es un
mapa **afín**, y la función media contaminaría la base recuperada por
`base_en_grilla`, la matriz de Gram y las autofunciones.

In [ ]:
NB_ELEGIDO  = 8     # ← decisión del analista
ORD_ELEGIDO = 3

print(f"Base elegida : n_basis={NB_ELEGIDO}, order={ORD_ELEGIDO}")
print(f"Sugerido GCV : n_basis={nb_best}, order={ord_best}"
      + ("   (coinciden)" if (NB_ELEGIDO, ORD_ELEGIDO) == (nb_best, ord_best)
         else "   ← DIFIERE de la sugerencia; justificar en la tesis"))

fr = FunctionalRepresentation(method="bspline", n_basis=NB_ELEGIDO,
                              order=ORD_ELEGIDO, center=False)
fr.fit(X_train, grilla)                       # sólo train
THETA       = fr.transform(X, grilla)         # (T, K) serie completa
THETA_train = THETA[idx_train]
print(f"THETA {THETA.shape}  (train={T0}, test={T - T0})")

# Error de la base B-spline, ANTES de cualquier truncamiento FPCA, por bloque.
_Xrec = fr.reconstruct(THETA)
_mise_bspline      = float((((X_true - _Xrec) ** 2) @ w_quad).mean())
_mise_bspline_tr   = float((((X_true[idx_train] - _Xrec[idx_train]) ** 2) @ w_quad).mean())
_mise_bspline_te   = float((((X_true[idx_test]  - _Xrec[idx_test])  ** 2) @ w_quad).mean())
print(f"\nMISE de la base B-spline contra la curva verdadera:")
print(f"  global {_mise_bspline:.6f}   train {_mise_bspline_tr:.6f}   "
      f"test {_mise_bspline_te:.6f}   razón test/train "
      f"{_mise_bspline_te / max(_mise_bspline_tr, 1e-15):.2f}x")
print("   La base B-spline se ajusta en train pero es un espacio LINEAL fijo: "
      "si esta\n   razón crece mucho, parte de la obsolescencia que §9 de 16_04 "
      "atribuye al FPCA\n   viene en realidad de aquí. Se reporta por separado.")

plot_fts_functional(
    X, grilla, fr=fr, highlight_idx=highlight_idx, separator_every=5,
    title=f"Algoritmo 6 — repr. B-spline (n_basis={NB_ELEGIDO}, order={ORD_ELEGIDO})",
    save_path=str(PATHS["out_report"] / "06_fts_funcional_bspline.png"))
plt.show()

guardar_representacion(PATHS, fr, THETA,
    extra={"T0": int(T0), "prop_train": float(PROP_TRAIN), "ajustado_en": "train",
           "center": bool(fr.center), "n_basis": int(NB_ELEGIDO),
           "order": int(ORD_ELEGIDO), "n_basis_gcv": nb_best, "order_gcv": ord_best,
           "mise_bspline_vs_verdadera": _mise_bspline,
           "mise_bspline_train": _mise_bspline_tr,
           "mise_bspline_test": _mise_bspline_te})
print("[functional] functional_representation.pkl + theta.csv + fr_config.json")

### 3.3 FPCA generalizado, y **la obsolescencia de la base**

La base B-spline no es ortonormal, de modo que la Gram $W\neq I$ y la
descomposición correcta resuelve el problema propio generalizado
$(W^{1/2}S_\theta W^{1/2})z=\lambda z$. Se ajusta **sólo con entrenamiento**,
que aquí no es una precaución metodológica sino **el objeto de estudio**: la
base se ajusta sobre el régimen viejo y se aplica sobre el nuevo.

La tabla añade dos columnas propias del escenario:

- `var_ratio_test`: la varianza que cada componente explica **en el bloque de
  prueba**, calculada proyectando el test sobre las autofunciones ajustadas en
  entrenamiento. Si la base siguiera vigente, coincidiría con `var_ratio`; el
  reordenamiento del espectro hace que no coincida.
- `razon_test_train`: el cociente entre ambas. Es la cifra que cuantifica la
  obsolescencia componente a componente, y en las dos componentes intercambiadas
  debe apartarse claramente de 1.

In [ ]:
Phi  = base_en_grilla(fr, THETA.shape[1])        # (G, K)
fpca = FPCA_L2().fit(THETA_train, Phi, grilla)

_ver = fpca.verificar(THETA_train, fr=fr)
print("Verificación FPCA_L2 (entrenamiento):")
for k, v in _ver.items():
    print(f"  {k:34s} = {v:.3e}" if isinstance(v, float) else f"  {k:34s} = {v}")

cond = _ver["cond_W"]
nota = ("← MUY ALTO: reduzca n_basis" if cond > 1e10 else
        "← alto: vigile las componentes menores" if cond > 1e6 else "(sano)")
print(f"\ncond(W) = {cond:.3e}  {nota}")

assert _ver["todo_ok"], ("Las identidades del FPCA generalizado no se cumplen. "
                         "Si falla err_linealidad_reconstruct_rel, revise center=False.")

In [ ]:
# ── Varianza en train contra varianza en test, sobre la MISMA base ───────────
K_FPCA = fpca.evals.size
S_full = (THETA - fpca.mu_theta) @ (fpca.W @ fpca.B_full)     # (T, K) todos los scores
S_a    = S_full[idx_train]
ar1    = (S_a[1:] * S_a[:-1]).sum(0) / np.clip((S_a[:-1] ** 2).sum(0), 1e-12, None)

var_tr = S_a.var(axis=0)
var_te = S_full[idx_test].var(axis=0)
vr_tr  = var_tr / var_tr.sum()
vr_te  = var_te / var_te.sum()
razon  = var_te / np.clip(var_tr, 1e-15, None)

# Alineación con la base del generador, calculada SOBRE ENTRENAMIENTO.
A_tr = A_coef[idx_train]
corr_AF = np.zeros((K_FPCA, J_GEN))
for k in range(K_FPCA):
    for j in range(J_GEN):
        _s, _a = S_a[:, k], A_tr[:, j]
        if _s.std() < 1e-12 or _a.std() < 1e-12:
            corr_AF[k, j] = 0.0
        else:
            corr_AF[k, j] = abs(float(np.corrcoef(_s, _a)[0, 1]))
fourier_de_fpc = np.argmax(corr_AF, axis=1) + 1        # base-1
corr_max       = corr_AF.max(axis=1)

VAR_TARGET  = 0.95
M_SUGERIDO  = fpca.seleccionar_M(VAR_TARGET)

tabla_fpca = pd.DataFrame({
    "componente": np.arange(1, K_FPCA + 1),
    "autovalor": fpca.evals, "var_ratio": fpca.var_ratio, "var_acum": fpca.var_cum,
    "ar1_propio": ar1,
    "var_ratio_test": vr_te, "razon_test_train": razon,
    "fourier_alineada": fourier_de_fpc, "corr_alineacion": corr_max,
})
display(tabla_fpca.head(min(15, K_FPCA)).style.format(
    {"autovalor": "{:.4e}", "var_ratio": "{:.4%}", "var_acum": "{:.4%}",
     "ar1_propio": "{:+.3f}", "var_ratio_test": "{:.4%}",
     "razon_test_train": "{:.3f}", "corr_alineacion": "{:.4f}"})
    .background_gradient(subset=["var_ratio"], cmap="YlGn")
    .background_gradient(subset=["razon_test_train"], cmap="coolwarm",
                         vmin=0, vmax=2)
    .set_caption("La MISMA base, evaluada en los dos bloques · "
                 "razon_test_train lejos de 1 = obsolescencia"))

print(f"\nK disponibles: {K_FPCA}   ·   sugerencia (var >= {VAR_TARGET:.0%}): "
      f"M = {M_SUGERIDO}")
print(f"\nrazón var(test)/var(train) por componente FPCA:")
for k in range(min(6, K_FPCA)):
    marca = ""
    if fourier_de_fpc[k] in (_i1, _i2):
        marca = f"  <- alineada con Fourier {fourier_de_fpc[k]} (intercambiada)"
    print(f"  FPC {k+1}: {razon[k]:.3f}{marca}")

# La verificación que decide si el escenario sobrevive al pipeline: las
# componentes alineadas con las intercambiadas deben moverse en direcciones
# OPUESTAS. Una baja y la otra sube.
_k_i1 = int(np.argmax(corr_AF[:, _i1 - 1]))
_k_i2 = int(np.argmax(corr_AF[:, _i2 - 1]))
print(f"\nFourier {_i1} → FPC {_k_i1 + 1} (razón {razon[_k_i1]:.3f})")
print(f"Fourier {_i2} → FPC {_k_i2 + 1} (razón {razon[_k_i2]:.3f})")
assert _k_i1 != _k_i2, "Ambas componentes de Fourier se alinean con la misma FPC."
assert (razon[_k_i1] - 1.0) * (razon[_k_i2] - 1.0) < 0, (
    "Las dos componentes intercambiadas no se mueven en direcciones opuestas "
    "entre train y test: el reordenamiento no sobrevivió al pipeline y el "
    "escenario no tendría contenido.")
print("\n  El reordenamiento SÍ sobrevive al pipeline: una componente pierde "
      "varianza y\n  la otra la gana. La base ajustada en entrenamiento ya no "
      "ordena correctamente\n  el bloque de prueba.")

plot_fpca_scree(fpca.evals, fpca.var_cum, M_SUGERIDO, var_target=VAR_TARGET,
                save_path=str(PATHS["out_report"] / "07_fpca_scree.png"))
plt.show()

In [ ]:
# Figura propia: la misma base evaluada en los dos bloques.
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
_n  = min(8, K_FPCA)
_kx = np.arange(1, _n + 1)
_ancho = 0.38

axes[0].bar(_kx - _ancho/2, vr_tr[:_n], _ancho, color="#2c3e50", alpha=0.85,
            label="entrenamiento")
axes[0].bar(_kx + _ancho/2, vr_te[:_n], _ancho, color="#e67e22", alpha=0.85,
            label="prueba")
axes[0].set_yscale("log"); axes[0].set_xticks(_kx)
axes[0].set_xlabel("componente FPCA"); axes[0].set_ylabel("var_ratio")
axes[0].legend(fontsize=8)
axes[0].set_title("La MISMA base, los dos bloques", fontsize=10)

_colr = ["#c0392b" if fourier_de_fpc[k] in (_i1, _i2) else "#7f8c8d"
         for k in range(_n)]
axes[1].bar(_kx, razon[:_n], color=_colr, alpha=0.85)
axes[1].axhline(1.0, color="k", ls="--", lw=1.2)
axes[1].set_xticks(_kx); axes[1].set_xlabel("componente FPCA")
axes[1].set_ylabel("var(test) / var(train)")
axes[1].set_title("Obsolescencia por componente (rojo = intercambiadas)",
                  fontsize=10)

fig.suptitle("Escenario 6 — la base ajustada en entrenamiento deja de ordenar "
             "el bloque de prueba", fontsize=12)
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "12_obsolescencia_base.png", dpi=150,
            bbox_inches="tight")
plt.show()

### 3.4 `[CONFIG]` Componentes FPCA retenidas

`M_FPCA = 4` es el **invariante del estudio**, igual que en las corridas 11 a
15. A diferencia del Algoritmo 5, aquí el valor de $M$ no decide si el escenario
prueba algo: el reordenamiento afecta a las dos componentes **dominantes**, que
están dentro del truncamiento con cualquier $M\geq 2$. El mecanismo no está en
qué se descarta sino en que **la base misma deja de ser la correcta**.

In [ ]:
M_FPCA = 4   # ← INVARIANTE del estudio, igual que las corridas 11 a 15.

assert 1 <= M_FPCA <= fpca.evals.size, f"M_FPCA fuera de [1, {fpca.evals.size}]."
fpca.set_M(int(M_FPCA))
M_fpca = fpca.M

Psi_grid, mu_grid = fpca.Psi_grid, fpca.mu_grid
SCORES       = fpca.transform(THETA)       # (T, M) — base ajustada en train
SCORES_train = SCORES[idx_train]
SCORES_test  = SCORES[idx_test]

print(f"M = {M_fpca}   var. explicada (en train) = {fpca.var_cum[M_fpca-1]:.4%}")
print(f"[train] max|media xi| = {np.abs(SCORES_train.mean(0)).max():.2e}   (aprox 0)")
print(f"[test]  max|media xi| = {np.abs(SCORES_test.mean(0)).max():.3f}")
print(f"[test]  var xi / lambda = "
      f"{np.array2string(SCORES_test.var(0, ddof=1) / fpca.lambdas, precision=3)}")
print("   ← ESTA es la firma del escenario: en un proceso estacionario esas "
      "razones\n     rondarían 1. Aquí las dos primeras se apartan porque el "
      "espectro se\n     reordenó y la base ya no lo refleja.")

# Ambas componentes intercambiadas están dentro del truncamiento: el mecanismo
# no depende de M.
assert max(_k_i1, _k_i2) < M_fpca, (
    f"Alguna de las componentes intercambiadas (FPC {_k_i1+1}, {_k_i2+1}) quedó "
    f"fuera del truncamiento a M={M_fpca}: el escenario perdería su rasgo.")
print(f"\n  Las dos componentes intercambiadas (FPC {_k_i1+1} y {_k_i2+1}) están "
      f"dentro de\n  las {M_fpca} retenidas: el mecanismo del escenario no "
      f"depende del valor de M.")

## 4. Datasets AR($p$) sobre los scores

### 4.1 Estandarización

El estandarizador se ajusta **sólo con train** y registra `n_ajuste` para que
esa disciplina sea auditable desde el artefacto y no una promesa del notebook.

In [ ]:
scores_standardizer = DataStandardizer(method="zscore_column", ddof=0)
scores_standardizer.fit(SCORES_train, etiqueta=f"train[1:{T0}]")

_chk = scores_standardizer.verificar_ajuste(T0)
print(f"[holdout] ajustado con {_chk['n_ajuste']} filas = T0 "
      f"({_chk['etiqueta_ajuste']}) → ok={_chk['ajuste_ok']}")

SCORES_STD       = scores_standardizer.transform(SCORES)
SCORES_STD_train = SCORES_STD[idx_train]
SCORES_STD_test  = SCORES_STD[idx_test]

print(f"\nSCORES_STD {SCORES_STD.shape}")
print(f"  [train] max|media| = {np.abs(SCORES_STD_train.mean(0)).max():.2e}  (aprox 0)")
print(f"  [train] max|std-1| = {np.abs(SCORES_STD_train.std(0) - 1).max():.2e}  (aprox 0)")
print(f"  [test]  media = {np.array2string(SCORES_STD_test.mean(0), precision=3)}")
print(f"  [test]  std   = {np.array2string(SCORES_STD_test.std(0),  precision=3)}")
print("\n   La std del bloque de prueba se aparta de 1 POR DISEÑO: el "
      "estandarizador se\n   ajustó sobre el régimen viejo. En las corridas 11 "
      "a 15 esa desviación era\n   ruido muestral; aquí es la medición.")

guardar_estandarizador(PATHS, scores_standardizer)
_res = guardar_fpca(PATHS, fpca, SCORES, SCORES_STD=SCORES_STD,
                    meta_extra={"T0": int(T0)})
print(f"\n[functional] artefactos FPCA · cond_W = {_res['meta']['cond_W']:.3e}")

plot_diagnostico_estandarizacion(
    SCORES_train, SCORES_STD_train, np.arange(1, M_fpca + 1),
    labels=("Scores xi (escala lambda)", "Scores xi estandarizados"),
    title="estadísticas por componente FPCA",
    save_path=str(PATHS["out_report"] / "08_diagnostico_estandarizacion.png"))
plt.show()

### 4.2 Diagnóstico de rezagos (sólo train)

In [ ]:
N_LAGS_MAX = 3
T_theta, K_total = SCORES_STD_train.shape

def _spearman(Y, Xm):
    """Spearman columna a columna vía rangos (pandas, sin scipy)."""
    Yc = pd.DataFrame(Y).rank().to_numpy(); Yc = Yc - Yc.mean(0)
    Xc = pd.DataFrame(Xm).rank().to_numpy(); Xc = Xc - Xc.mean(0)
    return (Yc.T @ Xc) / np.outer(np.sqrt((Yc**2).sum(0)), np.sqrt((Xc**2).sum(0)))

corr_p = np.zeros((K_total, K_total * N_LAGS_MAX))
corr_s = np.zeros_like(corr_p)
col_labels = []
y_block = SCORES_STD_train[N_LAGS_MAX:, :]

for lag in range(1, N_LAGS_MAX + 1):
    x_block = SCORES_STD_train[N_LAGS_MAX - lag : T_theta - lag, :]
    sp = _spearman(y_block, x_block)
    for j in range(K_total):
        c = (lag - 1) * K_total + j
        for k in range(K_total):
            corr_p[k, c] = np.corrcoef(y_block[:, k], x_block[:, j])[0, 1]
        corr_s[:, c] = sp[:, j]
        col_labels.append(rf"$\xi_{{t-{lag},{j+1}}}$")

row_labels = [rf"$\xi_{{t,{k+1}}}$" for k in range(K_total)]
band = 1.96 / np.sqrt(len(y_block))

for M_corr, nombre, arch in ((corr_p, "Pearson", "09a"), (corr_s, "Spearman", "09b")):
    plot_rezagos_heatmap(
        M_corr, col_labels, row_labels,
        title=f"{nombre} — respuesta($t$) vs rezagos 1..{N_LAGS_MAX}",
        n_lags_max=N_LAGS_MAX, K_total=K_total, band=band, vclip=0.6,
        save_path=str(PATHS["out_report"] / f"{arch}_rezagos_{nombre.lower()}.png"))
    plt.show()

print(f"correlación de cada score con su PROPIO rezago 1 "
      f"(objetivo: phi = {SIM_CFG.phi_comun} en TODAS):")
for k in range(K_total):
    print(f"  FPC {k+1}: {corr_p[k, k]:+.4f}")
_fuera = np.abs(corr_p[:, :K_total]).copy()
np.fill_diagonal(_fuera, 0.0)
print(f"\nmáx |corr| CRUZADA a rezago 1: {_fuera.max():.4f}   "
      f"(banda de ruido {band:.4f})")
print("   El generador no tiene dependencia cruzada; cualquier valor por encima "
      "de la\n   banda es ruido muestral. Contrastar con el Algoritmo 5: allí "
      "sólo UNA entrada\n   de la diagonal estaba poblada; aquí deben estarlo "
      "todas.")

### 4.3 `[CONFIG]` Orden AR y construcción de los datasets

Los rezagos del primer origen de prueba vienen del final del bloque de
entrenamiento: son observaciones pasadas disponibles en cada origen, de modo
que su uso es el condicionamiento de la predicción a $h=1$, no fuga.

In [ ]:
N_LAGS        = 1
COMPONENT_IDX = list(range(SCORES_STD.shape[1]))   # base-0; los nombres usan idx+1

n_components = len(COMPONENT_IDX)
n_train_eff  = T0 - N_LAGS
n_test_eff   = T - T0

assert T0 > N_LAGS and len(set(COMPONENT_IDX)) == n_components

cov_names = [f"fpc_{COMPONENT_IDX[j] + 1}_lag{lag}"
             for lag in range(1, N_LAGS + 1)
             for j in range(n_components)]

print(f"componentes : {n_components} → índices {COMPONENT_IDX}")
print(f"N_LAGS      : {N_LAGS}   ·   p = {len(cov_names)} covariables")
print(f"n_train_eff : {n_train_eff}   n_test_eff : {n_test_eff}")
print(f"cov_names   : {cov_names}")

In [ ]:
SCORES_sel = SCORES_STD[:, COMPONENT_IDX]

def _dataset_bloque(k, t_ini, t_fin):
    """Respuesta en t en [t_ini, t_fin) y predictores en t-1 … t-N_LAGS."""
    t_idx  = np.arange(t_ini, t_fin)
    X_cols = np.hstack([SCORES_sel[t_idx - lag, :] for lag in range(1, N_LAGS + 1)])
    return pd.DataFrame(np.column_stack([SCORES_sel[t_idx, k], X_cols]),
                        columns=[f"fpc_{COMPONENT_IDX[k] + 1}"] + cov_names)

dfs_train = {k: _dataset_bloque(k, N_LAGS, T0) for k in range(n_components)}
dfs_test  = {k: _dataset_bloque(k, T0,     T)  for k in range(n_components)}

manifest = {
    "scores_scale":  "standardized_zscore_ddof0",
    "n_components":  n_components,
    "n_lags":        int(N_LAGS),
    "component_idx": [int(i) for i in COMPONENT_IDX],
    "cov_names":     cov_names,
    "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
    "n_train_eff": int(n_train_eff), "n_test_eff": int(n_test_eff),
    "ajuste_en": "train",
}
guardar_datasets_ar(PATHS, dfs_train, dfs_test, manifest)
print(f"[functional] {2*n_components} datasets + datasets_manifest.json")
for k in range(n_components):
    print(f"  fpc_{COMPONENT_IDX[k]+1}: train {dfs_train[k].shape} · test {dfs_test[k].shape}")

#### Persistencia de la alineación FPCA ↔ generador

Ésta es la tabla que `16_03 §6.1` lee para declarar `VERDAD` y que `16_04` usa
en su sección de obsolescencia. Se escribe **después** de fijar `M`.

In [ ]:
alineacion_df = pd.DataFrame({
    "fpc":              np.arange(1, K_FPCA + 1),          # base-1
    "fourier_alineada": fourier_de_fpc,                    # base-1
    "corr_alineacion":  corr_max,
    "var_ratio_train":  vr_tr,
    "var_ratio_test":   vr_te,
    "razon_test_train": razon,
    "ar1_score":        ar1,
    "phi_generador":    np.array([PHIS[j - 1] for j in fourier_de_fpc]),
    "intercambiada":    np.isin(fourier_de_fpc, [_i1, _i2]),
    "retenida":         np.arange(1, K_FPCA + 1) <= M_fpca,
})
alineacion_df.to_csv(PATHS["out_report"] / "10_alineacion_fpca_generador.csv",
                     index=False)
print(f"[out_report] 10_alineacion_fpca_generador.csv  {alineacion_df.shape}")
display(alineacion_df.head(min(10, K_FPCA)).style.format({
    "corr_alineacion": "{:.4f}", "var_ratio_train": "{:.4%}",
    "var_ratio_test": "{:.4%}", "razon_test_train": "{:.3f}",
    "ar1_score": "{:+.4f}", "phi_generador": "{:+.2f}"}))

# Verificación numérica contra la fuente, no contra la memoria.
assert np.allclose(alineacion_df["phi_generador"].to_numpy(),
                   [PHIS[j - 1] for j in fourier_de_fpc]), \
    "phi_generador no coincide con internos['coeficientes_ar']."
assert np.allclose(PHIS, SIM_CFG.phi_comun), \
    "El generador no tiene phi común: revisar la configuración."
print(f"\n  Todos los phi_j valen {SIM_CFG.phi_comun}: la predictibilidad NO "
      f"varía entre componentes,\n  de modo que el escenario aísla el efecto "
      f"del reordenamiento del espectro.")

## 5. `[CONFIG]` Hiperparámetros y configuración MCMC

Esto es el **contrato con MATLAB**: `psbp_fd_iteracion.m` lee estos valores del
JSON y no los tiene escritos a mano, incluida `seed_base`.

Ojo con la doble acepción de `M`: aquí, dentro de `mcmc_config`, es el tamaño
de la grilla de localización $G^*$ del stick-breaking, **no** el número de
componentes FPCA. `N` es el truncamiento del número de átomos.

In [ ]:
MCMC_CONFIG = {"nsim": 2000, "burn": 500, "N": 35, "M": 35}
N_CHAINS    = 3

print(f"MCMC_CONFIG : {MCMC_CONFIG}")
print(f"N_CHAINS    : {N_CHAINS} cadenas por componente "
      f"→ {N_CHAINS * n_components} jobs en MATLAB")
print(f"Draws posteriores por score: "
      f"({MCMC_CONFIG['nsim']} - {MCMC_CONFIG['burn']}) x {N_CHAINS} = "
      f"{(MCMC_CONFIG['nsim'] - MCMC_CONFIG['burn']) * N_CHAINS}")
print(f"\nN = {MCMC_CONFIG['N']} átomos. El bloque de ENTRENAMIENTO es "
      f"gaussiano, lineal y\nhomogéneo —el quiebre está en t* = T0—, de modo "
      f"que cabe esperar una ocupación\nbaja, parecida a la de la corrida 15 "
      f"(ver 16_03 §4). Que la posterior no vea el\nquiebre es parte del "
      f"diseño.")

In [ ]:
# Priors globales
HP_GLOBAL = {"atau": 2.0, "btau": 0.5, "ag": 2.0, "bg": 0.5,
             "mumu": 0.0, "taumu": 1.0, "pwj": 0.5}

# Priors por tipo de covariable: (apij, bpij, mupsij, taupsij)
HP_BY_TYPE = {
    "own_lag1":  (9.0, 1.0, 0.0, 1.0),   # E[pi] = 0.90 — el propio rezago 1
    "cross_lag": (1.0, 1.0, 0.0, 1.0),   # E[pi] = 0.50 — los cruzados
}

def _clasificar(nombre, k_modelo):
    return ("own_lag1" if nombre == f"fpc_{COMPONENT_IDX[k_modelo] + 1}_lag1"
            else "cross_lag")

HYPERPARAMS_LIST = []
for k in range(n_components):
    tipos = [_clasificar(nm, k) for nm in cov_names]
    vals  = np.array([HP_BY_TYPE[t] for t in tipos], dtype=float)   # (p, 4)
    HYPERPARAMS_LIST.append({**HP_GLOBAL,
        "apij": vals[:, 0], "bpij": vals[:, 1],
        "mupsij": vals[:, 2], "taupsij": vals[:, 3]})

for k in range(n_components):
    hp = HYPERPARAMS_LIST[k]
    print(f"\nComponente k={k}  (fpc_{COMPONENT_IDX[k]+1})")
    print(f"  {'variable':<22} {'tipo':<11} {'apij':>6} {'bpij':>6} {'E[pi]':>7}")
    for j, nm in enumerate(cov_names):
        a, b = hp["apij"][j], hp["bpij"][j]
        marca = "  <-" if _clasificar(nm, k) == "own_lag1" else ""
        print(f"  {nm:<22} {_clasificar(nm, k):<11} {a:>6.1f} {b:>6.1f} "
              f"{a/(a+b):>7.3f}{marca}")

In [ ]:
hp_artifact = {
    "global":       HP_GLOBAL,
    "by_type":      HP_BY_TYPE,
    "mcmc_config":  MCMC_CONFIG,
    "n_iter":       N_CHAINS,
    "seed_scheme":  "seed_base + chain*9973 + k*31",
    "escenario_id": int(ESCENARIO_ID),
    "replica_id":   int(REPLICA_ID),
    "seed_base":    int(SEED),     # MATLAB la lee de aquí (ya no está hardcodeada)
    "scores_scale": "standardized_zscore_ddof0",
    "partition": {
        "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
        "n_train_eff": int(n_train_eff), "n_test_eff": int(n_test_eff),
        "train_files": [f"dataset_fpc_{COMPONENT_IDX[k]+1}_train.csv"
                        for k in range(n_components)],
        "test_files":  [f"dataset_fpc_{COMPONENT_IDX[k]+1}_test.csv"
                        for k in range(n_components)],
    },
    "hyperparams_list": [
        {"component_k": k, "fpc_idx": int(COMPONENT_IDX[k] + 1),
         "hyperparams": {key: (v.tolist() if isinstance(v, np.ndarray) else v)
                         for key, v in HYPERPARAMS_LIST[k].items()}}
        for k in range(n_components)
    ],
}
guardar_hiperparametros(PATHS, hp_artifact)
print(f"OK  hyperparameters.json → {PATHS['out_artefact']}")

## 6. Configuración de evaluación, líneas base y verificación del contrato

`objetivo_evaluacion` y `modo_residuo` se declaran aquí porque cambian el
significado de toda la evaluación y no deben quedar como una decisión implícita
del notebook `_04`.

In [ ]:
eval_config = {
    "scheme":      "holdout_temporal",
    "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
    "horizons":    [1],
    "n_lags":      int(N_LAGS),
    "scores_scale": "standardized_zscore_ddof0",
    "objetivo_evaluacion": "curva_verdadera",
    "modo_residuo":        "ninguno",
    "nivel_credibilidad":  0.95,
    "ventana_movil": {"w": [10, 20, 40], "paso": 1, "solapadas": True},
    "metrics_scores": ["RMSE", "R2", "razon_dispersion"],
    "metrics_curvas": ["MISE", "RMSE_funcional"],
    "metrics_dist":   ["CRPS", "energy_score", "cobertura_95", "PIT"],
    # Eje 2: calibración condicional al estado VERDADERO. Aquí es DISCRETO.
    "estratificacion": {
        "variable":    "quiebre_idx",
        "descripcion": "régimen de covarianza del generador (0 = pre, 1 = post)",
        "fuente":      "reports/.../10_estado_quiebre.csv",
        "respaldo":    f"raw/escenario_{ESCENARIO_ID}.npz::interno_trayectoria_espectro",
        "n_estratos":  2,
        "etiquetas":   ["pre-quiebre", "post-quiebre"],
        "metodo":      "estado discreto del generador, sin cuantilizar",
        "advertencia": (
            "t* = T0, de modo que este estrato COINCIDE con la partición "
            "train/test. La cobertura condicional y la comparación train/test "
            "miden lo mismo. No es un defecto —es la tesis del escenario— pero "
            "hay que declararlo: el desplome de cobertura en el bloque de "
            "prueba es pérdida de VIGENCIA DE LA BASE, no pérdida de "
            "generalización."
        ),
    },
    # Propio del Bloque 2: la representación es el objeto de estudio.
    "representacion": {
        "bloque_del_anexo":     2,
        "J_generador":          int(J_GEN),
        "base_generador":       "fourier ortonormal en L2",
        "phi_comun":            float(SIM_CFG.phi_comun),
        "espectro_inicial":     [float(l) for l in LAM0],
        "espectro_final":       [float(l) for l in LAM1],
        "indices_intercambio":  [int(_i1), int(_i2)],
        "t_quiebre":            int(TQ),
        "t_quiebre_default_del_codigo": 270,
        "periodos_adaptacion":  int(N_ADAPT),
        "M_retenidas":          int(M_fpca),
        "fpc_intercambiadas":   [int(_k_i1 + 1), int(_k_i2 + 1)],
        "razon_var_test_train": [float(r) for r in razon[:M_fpca]],
        "mise_bspline_vs_verdadera": float(_mise_bspline),
        "mise_bspline_train":   float(_mise_bspline_tr),
        "mise_bspline_test":    float(_mise_bspline_te),
        "fuente":               "reports/.../10_alineacion_fpca_generador.csv",
        "advertencia": (
            "El resultado de este escenario NO discrimina entre "
            "especificaciones dinámicas: la obsolescencia de la base alcanza "
            "por igual a todo método sobre la misma representación. Acota el "
            "alcance de la reducción de dimensión."
        ),
    },
}
guardar_config_evaluacion(PATHS, eval_config)
print("[out_artefact] eval_config.json")
for k, v in eval_config.items():
    print(f"  {k:22s}: {v}")

In [ ]:
# Líneas base a h=1: el piso que el PSBP-FD debe superar.
baselines_df = tabla_baselines(SCORES_STD, T0, estandarizador=scores_standardizer,
                               fpca=fpca, X_obs=X_true, tau=grilla, h=1)
baselines_df.to_csv(PATHS["out_report"] / "30_baselines_test.csv")
display(baselines_df.style.format("{:.4f}", na_rep="—")
        .set_caption("Líneas base — bloque de prueba, h=1, contra la curva VERDADERA"))

In [ ]:
informe = verificar_contrato(PATHS)
print(f"contrato_ok = {informe['contrato_ok']}   "
      f"(M={informe['M']}, K={informe['K']}, T0={informe['T0']}, "
      f"n_components={informe['n_components']})")
print(f"estandarizador ajustado con {informe.get('estandarizador_n_ajuste')} filas (T0={informe['T0']})")
print(f"verificación FPCA: todo_ok = {informe['verificacion_fpca']['todo_ok']}")
if not informe["contrato_ok"]:
    print("\nPROBLEMAS:")
    for p in informe.get("problemas", []):
        print(f"  - {p}")

print(f"\n{'='*66}\nListo. Siguiente paso, en MATLAB desde esta carpeta:\n"
      f"  >> psbp_fd_iteracion\n"
      f"EXPERIMENT_ID = {EXPERIMENT_ID}\n"
      f"t* = {TQ} = T0  ·  el estrato del eje 2 coincide con la partición\n"
      f"{'='*66}")